# Match the Privacy CPC Codes and Build the Data Files

| Item | Description |
|---|---|
| Goal | The two data files that the X1.2 notebooks read. `sample_firm_privacy_patents.csv` is one row per sample-firm patent and listed privacy code, filing years 1999 to 2021, with the patent's full CPC set. `firm_topic_relevance.csv` is the 1,058 sample firms with the main topic relevance score of T3.4.1. |
| Input | `sample_firm_patents.parquet` from X1.1.1 (the X1.1 `Processed` folder on scratch); `Match_patent_cpc_2025.zip`, the CPC match file of the KPSS repository (the X1.1 `Raw` folder on scratch); `Data/privacy_cpc_codes.xlsx`, the 72-code privacy list, with `Data/privacy_cpc_codes_MM_102926.xlsx` (the five focal codes) and `Data/privacy_cpc_codes_clustered_by_claude.csv` (the 72 codes in four groups) checked against it; `table_firm_relevance_scores.csv` from T3.4.1 under `R - Results/tables`. |
| Output | `Data/sample_firm_privacy_patents.csv` and `Data/firm_topic_relevance.csv` in the section's `Data` folder. On scratch, `Processed/sample_firm_patent_cpc_codes.parquet`: one row per patent-code pair for every sample-firm patent, all filing years. Under `R - Results/tables/.../X1.1.2 ...`: `table_privacy_patents_by_code.csv`, the 72 codes with their patent and firm counts. |
| Unit | A patent-code pair (`patent_num`, `privacy_cpc`) in the patent file; a firm (gvkey) of the all-years T3 sample in the relevance file. |
| Privacy code | A code of the 72-code list matched exactly to the patent's full CPC code, after removing the space the list writes between the class and the group (the KPSS file writes `G06F21/6245`, the list `G06F 21/6245`). A listed parent code does not pull in unlisted child codes. The five focal codes of X1.2 are among the 72. |
| Run | Run 2026-11-01 from the X1.1.1 run of the same day; the two data files reproduce the files committed on 2026-10-31 byte for byte. |

The KPSS data are to be cited as Kogan, L., Papanikolaou, D., Seru, A., and Stoffman, N. (2017), Technological Innovation, Resource Allocation, and Growth, *Quarterly Journal of Economics*, 132(2), 665 to 712, with the extended data and the CPC match file from the authors' GitHub repository.


## Step 1. Import packages


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


## Step 2. Define file locations and the period

All folders in one place: the clone root from the notebook's location, the scratch folder, the section's `Data` folder, and the T3.4.1 score table under `R - Results/tables`.


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("X1 - Example of Research Enabled by LLM Pipeline/X1.2 - Notebooks for Matching and Aggregating Patent Data")
NOTEBOOK_NAME = "X1.2.2 - Patents - Match the Privacy CPC Codes and Build the Data Files"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
DATA_FOLDER = PROJECT_FOLDER / "X1 - Example of Research Enabled by LLM Pipeline/X1.1 - Data for Matching and Patent Analysis"
PATENTS_FILE = PROCESSED_FOLDER / "sample_firm_patents.parquet"  # X1.1.1: one row per patent filed by a sample firm
CPC_ZIP = RAW_FOLDER / "Technological-Innovation-Resource-Allocation-and-Growth-Extended-Data/Match_patent_cpc_2025.zip"
CODES_FILE = DATA_FOLDER / "privacy_cpc_codes.xlsx"  # the 72-code privacy list
FIVE_CODES_FILE = DATA_FOLDER / "privacy_cpc_codes_MM_102926.xlsx"  # the five focal codes of X1.2
CLUSTERED_FILE = DATA_FOLDER / "privacy_cpc_codes_clustered_by_claude.csv"  # the 72 codes in four groups, read by X1.2.3
SCORES_FILE = PROJECT_FOLDER / "R - Results/tables/T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.4 - Topic Relevance - Score and Figures/T3.4.1 - Score - Cosine of the Firm Summary and the Policy Summary/table_firm_relevance_scores.csv"  # T3.4.1: one row per firm and sample
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME

SAMPLE = "all_years"  # the T3 sample whose scores are copied
PERIOD_START = 1999  # filing years kept in the patent file; later filings are not fully granted in the KPSS data
PERIOD_END = 2021
for folder in (PROCESSED_FOLDER, TABLE_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)
print(f"Data files: {DATA_FOLDER}")
print(f"Tables: {TABLE_FOLDER}")


Data files: /project2/njia_1089/acroh/projects/OS_LLM_TEMP/X1 - Example of Research Enabled by LLM Pipeline/Data
Tables: /project2/njia_1089/acroh/projects/OS_LLM_TEMP/R - Results/tables/X1 - Example of Research Enabled by LLM Pipeline/X1.1 - Patent Data - Sample Firm Patents and Privacy Codes/X1.1.2 - Patents - Match the Privacy CPC Codes and Build the Data Files


## Step 3. Load the privacy code lists

The 72-code list, one row per code with its CPC definition. The space between the class and the group is removed so that the codes match the KPSS spelling. The five focal codes of X1.2 and the four-group list of X1.2.3 must be subsets of it.


In [3]:
privacy_codes = pd.read_excel(CODES_FILE)
privacy_codes.columns = ["cpc_display", "definition"]
privacy_codes["cpc_display"] = privacy_codes["cpc_display"].str.strip()
privacy_codes["cpc"] = privacy_codes["cpc_display"].str.replace(" ", "", regex=False)  # "G06F 21/6245" -> "G06F21/6245", the KPSS spelling
privacy_codes["cpc_class"] = privacy_codes["cpc"].str[:4]
assert privacy_codes["cpc"].is_unique, "a code appears twice in the list"
five_codes = pd.read_excel(FIVE_CODES_FILE).iloc[:, 0].astype(str).str.strip().str.replace(" ", "", regex=False)
clustered_codes = pd.read_csv(CLUSTERED_FILE)["cpc_display"].str.strip().str.replace(" ", "", regex=False)
assert five_codes.isin(privacy_codes["cpc"]).all(), sorted(set(five_codes) - set(privacy_codes["cpc"]))
assert set(clustered_codes) == set(privacy_codes["cpc"]), "the clustered list and the code list differ"
print(f"Privacy CPC codes: {len(privacy_codes)}; by class: {privacy_codes['cpc_class'].value_counts().to_dict()}; "
      f"the five focal codes of X1.2, all listed: {', '.join(five_codes)}")
display(privacy_codes.head(5))


Privacy CPC codes: 72; by class: {'H04L': 30, 'G06F': 17, 'G06Q': 14, 'H04W': 11}; the five focal codes of X1.2, all listed: G06Q30/0269, G06Q30/0255, G06F21/6254, H04L67/535, G06F11/3438


,cpc_display,definition,cpc,cpc_class
0,H04L 63/0407,wherein the identity of one or more communicat...,H04L63/0407,H04L
1,H04L 63/0414,"during transmission, i.e. party's identity is ...",H04L63/0414,H04L
2,H04L 63/0421,"Anonymous communication, i.e. the party's iden...",H04L63/0421,H04L
3,H04L 61/2539,Hiding addresses; Keeping addresses anonymous,H04L61/2539,H04L
4,H04L 2209/42,"Anonymization, e.g. involving pseudonyms",H04L2209/42,H04L


## Step 4. Attach the CPC codes to the sample firms' patents

The CPC match file lists every patent's codes separated by semicolons. The codes of every sample-firm patent of X1.1.1, all filing years, are split into one row per patent-code pair and saved on scratch.


In [4]:
all_patents = pd.read_parquet(PATENTS_FILE, columns=["patent_num", "gvkey", "conm", "filing_date", "filing_year"])
cpc_match = pd.read_csv(CPC_ZIP, dtype={"patent_num": "int64", "cpc": "string"})
cpc_match = cpc_match.loc[cpc_match["patent_num"].isin(all_patents["patent_num"])]
codes = cpc_match.dropna(subset=["cpc"]).copy()
codes["cpc"] = codes["cpc"].astype(str).str.split(";")
codes = codes.explode("cpc")
codes["cpc"] = codes["cpc"].str.strip()
codes = codes.loc[codes["cpc"].ne("")]
codes["cpc_class"] = codes["cpc"].str[:4]
codes = codes.drop_duplicates(["patent_num", "cpc"]).sort_values(["patent_num", "cpc"]).reset_index(drop=True)
codes.to_parquet(PROCESSED_FOLDER / "sample_firm_patent_cpc_codes.parquet", index=False)
n_without_codes = int((~all_patents["patent_num"].isin(codes["patent_num"])).sum())
print(f"Sample-firm patents, all filing years: {len(all_patents):,}; patent-code pairs: {len(codes):,}; patents without any CPC code: {n_without_codes:,}")
display(codes.head(3))


Sample-firm patents, all filing years: 1,315,652; patent-code pairs: 8,446,709; patents without any CPC code: 758


,patent_num,cpc,cpc_class
0,5922498,G03G5/047,G03G
1,5922498,G03G5/0633,G03G
2,5922498,G03G5/142,G03G


## Step 5. Build the privacy patent file

One row per patent filed in the period and listed code it carries, with the firm, the filing date, and the patent's full CPC set in `cpc_all`. A patent with two listed codes has two rows. Saved to the `Data` folder.


In [5]:
in_period = all_patents.loc[all_patents["filing_year"].between(PERIOD_START, PERIOD_END)]
cpc_all = codes.groupby("patent_num")["cpc"].agg(";".join)  # the codes of a patent, sorted, in one string
matches = codes.loc[codes["cpc"].isin(privacy_codes["cpc"]), ["patent_num", "cpc"]].merge(in_period, on="patent_num", how="inner")
privacy_patents = pd.DataFrame({"patent_num": matches["patent_num"], "gvkey": matches["gvkey"].astype(str).str.zfill(6), "conm": matches["conm"],
                                "filing_date": matches["filing_date"].dt.strftime("%Y-%m-%d"), "cpc_all": matches["patent_num"].map(cpc_all),
                                "privacy_cpc": matches["cpc"]})
privacy_patents = privacy_patents.sort_values(["patent_num", "privacy_cpc"]).reset_index(drop=True)
privacy_patents.to_csv(DATA_FOLDER / "sample_firm_privacy_patents.csv", index=False)
never_matched = privacy_codes.loc[~privacy_codes["cpc"].isin(privacy_patents["privacy_cpc"]), "cpc_display"].tolist()
print(f"Patents filed in {PERIOD_START} to {PERIOD_END}: {len(in_period):,} by {in_period['gvkey'].nunique():,} firms; "
      f"privacy patents: {privacy_patents['patent_num'].nunique():,} by {privacy_patents['gvkey'].nunique():,} firms; patent-code rows: {len(privacy_patents):,}")
print(f"Listed codes that never occur on a sample-firm patent in the period: {len(never_matched)} ({', '.join(never_matched)})")
print(f"Saved: {DATA_FOLDER / 'sample_firm_privacy_patents.csv'}")
display(privacy_patents.head(3))


Patents filed in 1999 to 2021: 1,238,690 by 709 firms; privacy patents: 33,986 by 279 firms; patent-code rows: 47,225
Listed codes that never occur on a sample-firm patent in the period: 3 (G06Q 30/02011, G06Q 30/02023, G06Q 30/06311)
Saved: /project2/njia_1089/acroh/projects/OS_LLM_TEMP/X1 - Example of Research Enabled by LLM Pipeline/Data/sample_firm_privacy_patents.csv


,patent_num,gvkey,conm,filing_date,cpc_all,privacy_cpc
0,6061684,012141,MICROSOFT CORP,1999-01-29,G06F17/30221;G06F21/6218;G06F2221/2145;H04L63/...,G06F21/6218
1,6160501,006008,INTEL CORP,1999-05-11,G06F21/6209,G06F21/6209
2,6166688,006066,INTL BUSINESS MACHINES CORP,1999-03-31,G06F21/575;G06F21/81;G06F21/88;G06F2221/2111;G...,H04W12/08


## Step 6. Count the privacy patents by code

One row per listed code with the number of distinct patents and of firms; codes that never occur keep a zero.


In [6]:
by_code = (privacy_patents.groupby("privacy_cpc").agg(n_privacy_patents=("patent_num", "nunique"), n_firms_filing=("gvkey", "nunique"))
           .reindex(privacy_codes["cpc"]).fillna(0).astype(int).reset_index().rename(columns={"privacy_cpc": "cpc"}))
by_code = privacy_codes[["cpc_display", "cpc", "cpc_class", "definition"]].merge(by_code, on="cpc", how="left", validate="one_to_one")
by_code = by_code.sort_values(["n_privacy_patents", "cpc"], ascending=[False, True]).reset_index(drop=True)
by_code.to_csv(TABLE_FOLDER / "table_privacy_patents_by_code.csv", index=False)
print(f"Saved: {TABLE_FOLDER / 'table_privacy_patents_by_code.csv'} ({len(by_code)} codes)")
display(by_code.head(10))


Saved: /project2/njia_1089/acroh/projects/OS_LLM_TEMP/R - Results/tables/X1 - Example of Research Enabled by LLM Pipeline/X1.1 - Patent Data - Sample Firm Patents and Privacy Codes/X1.1.2 - Patents - Match the Privacy CPC Codes and Build the Data Files/table_privacy_patents_by_code.csv (72 codes)


,cpc_display,cpc,cpc_class,definition,n_privacy_patents,n_firms_filing
0,H04L 63/0428,H04L63/0428,H04L,"wherein the data content is protected, e.g. by...",4394,158
1,H04L 63/10,H04L63/10,H04L,for controlling access to devices or network r...,4135,140
2,G06F 21/6218,G06F21/6218,G06F,"to a system of files or objects, e.g. local or...",4057,132
3,H04W 12/08,H04W12/08,H04W,Access security,3086,111
4,H04W 4/029,H04W4/029,H04W,Location-based management or tracking services,2577,114
5,G06F 21/602,G06F21/602,G06F,Providing cryptographic facilities or services,2349,110
6,G06F 21/6245,G06F21/6245,G06F,"Protecting personal data, e.g. for financial o...",2124,107
7,H04W 12/02,H04W12/02,H04W,"Protecting privacy or anonymity, e.g. protecti...",1519,97
8,G06Q 30/0269,G06Q30/0269,G06Q,based on user profile or attribute,1430,78
9,G06F 21/62,G06F21/62,G06F,"Protecting access to data via a platform, e.g....",1318,89


## Step 7. Copy the topic relevance scores of the sample firms

The all-years rows of the T3.4.1 score table: the firm, its name, its number of filings, and the cosine of its firm summary with the policy summary. Saved to the `Data` folder under the column name the X1.2 notebooks read.


In [7]:
scores = pd.read_csv(SCORES_FILE, dtype={"gvkey": str})
relevance = (scores.loc[scores["sample"].eq(SAMPLE), ["gvkey", "conm", "n_filings", "score"]]
             .rename(columns={"score": "cosine_policy_summary_firm_summary"}).reset_index(drop=True))
assert relevance["gvkey"].is_unique
relevance.to_csv(DATA_FOLDER / "firm_topic_relevance.csv", index=False)
print(f"Saved: {DATA_FOLDER / 'firm_topic_relevance.csv'} ({len(relevance):,} firms; score from {relevance['cosine_policy_summary_firm_summary'].min():.3f} "
      f"to {relevance['cosine_policy_summary_firm_summary'].max():.3f})")
display(relevance.head(3))


Saved: /project2/njia_1089/acroh/projects/OS_LLM_TEMP/X1 - Example of Research Enabled by LLM Pipeline/Data/firm_topic_relevance.csv (1,058 firms; score from 0.403 to 0.621)


,gvkey,conm,n_filings,cosine_policy_summary_firm_summary
0,001013,ADC TELECOMMUNICATIONS INC,12,0.522551
1,001045,AMERICAN AIRLINES GROUP INC,22,0.534055
2,001075,PINNACLE WEST CAPITAL CORP,15,0.455298


## Step 8. Check the two files

Every patent belongs to a sample firm, no patent-code pair repeats, and the counts the X1.2 notebooks rely on are printed.


In [8]:
patent_file = pd.read_csv(DATA_FOLDER / "sample_firm_privacy_patents.csv", dtype={"gvkey": str})
relevance_file = pd.read_csv(DATA_FOLDER / "firm_topic_relevance.csv", dtype={"gvkey": str})
assert patent_file["gvkey"].isin(relevance_file["gvkey"]).all(), "a patent of a firm outside the sample"
assert patent_file.duplicated(["patent_num", "privacy_cpc"]).sum() == 0
print(f"sample_firm_privacy_patents.csv: {len(patent_file):,} rows, {patent_file['patent_num'].nunique():,} patents, {patent_file['gvkey'].nunique():,} firms, "
      f"{patent_file['privacy_cpc'].nunique()} codes, filing dates {patent_file['filing_date'].min()} to {patent_file['filing_date'].max()}")
print(f"firm_topic_relevance.csv: {len(relevance_file):,} firms, columns {list(relevance_file.columns)}")


sample_firm_privacy_patents.csv: 47,225 rows, 33,986 patents, 279 firms, 69 codes, filing dates 1999-01-05 to 2021-12-31
firm_topic_relevance.csv: 1,058 firms, columns ['gvkey', 'conm', 'n_filings', 'cosine_policy_summary_firm_summary']
